In [2]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm

# ==========================================
# 1. IMABlock & Trimodal Architecture
# ==========================================
class IMABlock(nn.Module):
    def __init__(self, d_query, d_target, num_heads=8):
        super(IMABlock, self).__init__()
        self.query_proj = nn.Linear(d_query, d_target)
        self.mha = nn.MultiheadAttention(embed_dim=d_target, num_heads=num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_target)

    def forward(self, query_cls, target_seq):
        q_projected = self.query_proj(query_cls) 
        attn_output, _ = self.mha(query=q_projected, key=target_seq, value=target_seq)
        output = self.layer_norm(q_projected + attn_output)
        return output 

In [3]:
class Trimodal_SSE_FT(nn.Module):
    def __init__(self, num_classes=7, d_speech=768, d_text=1024, d_video=256, fusion_dim=512):
        super(Trimodal_SSE_FT, self).__init__()
        
        # We only need Roberta here, since Audio and Video are pre-extracted!
        self.roberta = RobertaModel.from_pretrained("roberta-large")
        for param in self.roberta.parameters():
            param.requires_grad = False

        self.speech_cls_token = nn.Parameter(torch.randn(1, 1, d_speech))
        self.video_cls_token = nn.Parameter(torch.randn(1, 1, d_video))
        
        s_encoder_layer = nn.TransformerEncoderLayer(d_model=d_speech, nhead=8, batch_first=True)
        self.speech_self_attn = nn.TransformerEncoder(s_encoder_layer, num_layers=1)
        
        v_encoder_layer = nn.TransformerEncoderLayer(d_model=d_video, nhead=8, batch_first=True)
        self.video_self_attn = nn.TransformerEncoder(v_encoder_layer, num_layers=1)

        self.ima_s2t = IMABlock(d_query=d_speech, d_target=d_text)
        self.ima_s2v = IMABlock(d_query=d_speech, d_target=d_video)
        self.ima_t2s = IMABlock(d_query=d_text, d_target=d_speech)
        self.ima_t2v = IMABlock(d_query=d_text, d_target=d_video)
        self.ima_v2s = IMABlock(d_query=d_video, d_target=d_speech)
        self.ima_v2t = IMABlock(d_query=d_video, d_target=d_text)

        self.proj_s2t = nn.Linear(d_text, fusion_dim)
        self.proj_s2v = nn.Linear(d_video, fusion_dim)
        self.proj_t2s = nn.Linear(d_speech, fusion_dim)
        self.proj_t2v = nn.Linear(d_video, fusion_dim)
        self.proj_v2s = nn.Linear(d_speech, fusion_dim)
        self.proj_v2t = nn.Linear(d_text, fusion_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(fusion_dim * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pre_extracted_audio, input_text_ids, input_text_mask, pre_extracted_video):
        batch_size = pre_extracted_audio.size(0)

        # 1. Feature Extraction
        speech_out = pre_extracted_audio # Audio is already extracted!
        text_out = self.roberta(input_ids=input_text_ids, attention_mask=input_text_mask).last_hidden_state 
        video_out = pre_extracted_video  # Video is already extracted!

        # 2. Intra-Modal Summarization
        cls_s = self.speech_cls_token.expand(batch_size, -1, -1)
        speech_seq = torch.cat((cls_s, speech_out), dim=1)
        speech_seq = self.speech_self_attn(speech_seq)
        speech_cls = speech_seq[:, 0:1, :] 
        
        cls_v = self.video_cls_token.expand(batch_size, -1, -1)
        video_seq = torch.cat((cls_v, video_out), dim=1)
        video_seq = self.video_self_attn(video_seq)
        video_cls = video_seq[:, 0:1, :] 

        text_cls = text_out[:, 0:1, :] 

        # 3. Inter-Modality Attention (IMA)
        out_s2t = self.ima_s2t(speech_cls, text_out) 
        out_s2v = self.ima_s2v(speech_cls, video_seq) 
        out_t2s = self.ima_t2s(text_cls, speech_seq) 
        out_t2v = self.ima_t2v(text_cls, video_seq) 
        out_v2s = self.ima_v2s(video_cls, speech_seq) 
        out_v2t = self.ima_v2t(video_cls, text_out) 

        # 4. Dimension Projection & Hadamard Products
        speech_final = torch.mul(self.proj_s2t(out_s2t.squeeze(1)), self.proj_s2v(out_s2v.squeeze(1)))
        text_final   = torch.mul(self.proj_t2s(out_t2s.squeeze(1)), self.proj_t2v(out_t2v.squeeze(1)))
        video_final  = torch.mul(self.proj_v2s(out_v2s.squeeze(1)), self.proj_v2t(out_v2t.squeeze(1)))

        # 5. Concatenation & Classification
        combined_features = torch.cat((speech_final, text_final, video_final), dim=1)
        logits = self.classifier(combined_features) 
        return logits

In [4]:
# ==========================================
# 2. Dataset & Collate Function
# ==========================================
class MeldDataset(Dataset):
    def __init__(self, csv_path, video_dir, audio_dir):
        self.df = pd.read_csv(csv_path)
        self.video_dir = video_dir
        self.audio_dir = audio_dir
        
        # Initialize Text Tokenizer
        self.tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
        
        # MELD Label Mapping
        self.label_map = {'neutral': 0, 'joy': 1, 'sadness': 2, 'anger': 3, 'surprise': 4, 'fear': 5, 'disgust': 6}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']
        filename = f"dia{dia_id}_utt{utt_id}.pt"
        
        video_path = os.path.join(self.video_dir, filename)
        audio_path = os.path.join(self.audio_dir, filename)
        
        # Skip if missing files (due to face extraction failure)
        if not os.path.exists(video_path) or not os.path.exists(audio_path):
            return None

        # Load Pre-Extracted Tensors
        video_tensor = torch.load(video_path, map_location='cpu', weights_only=True) 
        audio_tensor = torch.load(audio_path, map_location='cpu', weights_only=True) 
        
        if audio_tensor.dim() == 3:
            audio_tensor = audio_tensor.squeeze(0) # Remove empty batch dim

        # Process Text on the fly
        text = str(row['Utterance'])
        tokens = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=128)
        text_ids = tokens['input_ids'].squeeze(0)
        text_mask = tokens['attention_mask'].squeeze(0)
        
        label = self.label_map[row['Emotion']]
        
        return {
            'audio': audio_tensor,
            'text_ids': text_ids,
            'text_mask': text_mask,
            'video': video_tensor,
            'label': torch.tensor(label, dtype=torch.long)
        }

def pad_collate_fn(batch):
    # Filter out missing records
    batch = [b for b in batch if b is not None]
    if len(batch) == 0: return None
    
    audios = [b['audio'] for b in batch]
    text_ids = torch.stack([b['text_ids'] for b in batch])
    text_masks = torch.stack([b['text_mask'] for b in batch])
    videos = [b['video'] for b in batch]
    labels = torch.stack([b['label'] for b in batch])
    
    # Pad variable-length videos/audio with zeros to make them the same length in the batch!
    padded_audios = pad_sequence(audios, batch_first=True)
    padded_videos = pad_sequence(videos, batch_first=True)
    
    return padded_audios, text_ids, text_masks, padded_videos, labels

In [ ]:
# ==========================================
# 3. Focal Loss & Training Setup
# ==========================================
import numpy as np
import torch.nn.functional as F

# The custom Focal Loss function
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha # The calculated Class weights
        self.gamma = gamma # Focusing parameter (2.0 is standard)

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-F.cross_entropy(inputs, targets, reduction='none'))
        # The focal loss formula!
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🚀 Initializing Trimodal Training on {device}...")
    
    model = Trimodal_SSE_FT(num_classes=7).to(device)
    
    print("Loading Training Dataset...")
    train_dataset = MeldDataset(
        csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\train_sent_emo.csv",
        video_dir=r"./meld_features/train",
        audio_dir=r"./meld_features_audio/train"
    )
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate_fn)
    
    # ---------------------------------------------------------
    # NEW: Calculate Class Weights from the Dataset!
    # ---------------------------------------------------------
    print("Calculating Class Weights for Focal Loss...")
    class_counts = train_dataset.df['Emotion'].value_counts()
    
    # Ensure they match the 0-6 index order exactly!
    ordered_emotions = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']
    counts_array = [class_counts.get(emo, 1) for emo in ordered_emotions] # get(emo, 1) prevents divide by zero
    
    # Inverse frequency formula: Weight = Total_Samples / Class_Samples
    total_samples = sum(counts_array)
    weights = [total_samples / c for c in counts_array]
    
    # Convert to PyTorch Tensor and normalize
    class_weights = torch.FloatTensor(weights).to(device)
    class_weights = class_weights / class_weights.sum() 
    print(f"Computed Weights: {class_weights.cpu().numpy()}")
    # ---------------------------------------------------------

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # NEW: Swap standard CrossEntropy for Focal Loss!
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    
    # Let the training begin!
    EPOCHS = 5
    for epoch in range(EPOCHS):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch in progress_bar:
            if batch is None: continue
            
            audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
            
            optimizer.zero_grad()
            logits = model(audios, text_ids, text_masks, videos)
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total:.4f}"})
            
        print(f"Epoch {epoch+1} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")

    # Save the trained model!
    torch.save(model.state_dict(), "trimodal_emotion_model_focal.pth")
    print("Model permanently saved as trimodal_emotion_model_focal.pth!")


In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize the blank architecture
print("Initializing model architecture...")
loaded_model = Trimodal_SSE_FT(num_classes=7).to(device)

# 2. Load the trained weights from your hard drive
# (Make sure this perfectly matches the name you saved it as!)
saved_model_path = "trimodal_emotion_model_focal.pth"
loaded_model.load_state_dict(torch.load(saved_model_path, map_location=device))

# 3. Switch to Evaluation Mode
# This is CRITICAL! It turns off Dropout and locks the weights so it doesn't try to learn during testing.
loaded_model.eval()

print(f"Trained model '{saved_model_path}' successfully loaded and ready for predictions, bros!")


Initializing model architecture...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trained model 'trimodal_emotion_model_focal.pth' successfully loaded and ready for predictions, bros!
